# Nykaa Domain Support Agent — Task 7: LangGraph Agent with Conditional Routing

## Goal
Build a LangGraph workflow that understands whether a customer is asking:
1. A **policy question** → search the recommended sentence-based RAG collection from Tasks 3–5.
2. An **order-status question** → call the Task 6 `check_order_status()` lookup function.

The graph contains four nodes:
- `classify_intent`
- `retrieve_policy`
- `lookup_order`
- `compose_response`

A genuine conditional edge after `classify_intent` sends the query to the correct tool. Both branches then connect to `compose_response`.

**Run this notebook from the same project folder** that contains:
- `orders.csv` or `orders_with_escalation_scores.csv`
- `chroma_db/` created in Task 3

This notebook uses deterministic, extractive `MOCK_LLM`-style response composition. It does not call a paid LLM API.

## 1. Install LangGraph

In [ ]:
%pip install -U langgraph

## 2. Imports and project paths

In [ ]:
from pathlib import Path
from typing import TypedDict, Optional, Literal
import re
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

from langgraph.graph import StateGraph, START, END

PROJECT_DIR = Path.cwd()
CHROMA_DIR = PROJECT_DIR / "chroma_db"
SCORED_ORDERS_PATH = PROJECT_DIR / "orders_with_escalation_scores.csv"
ORDERS_PATH = PROJECT_DIR / "orders.csv"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "nykaa_sentence_chunks"  # Recommended from Task 5 results
TOP_K = 3

print("Project directory:", PROJECT_DIR.resolve())
print("Chroma directory:", CHROMA_DIR.resolve())
print("Selected RAG collection:", COLLECTION_NAME)

if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        "chroma_db was not found. Run Task 3 in this same project directory."
    )

if SCORED_ORDERS_PATH.exists():
    ORDERS_INPUT = SCORED_ORDERS_PATH
elif ORDERS_PATH.exists():
    ORDERS_INPUT = ORDERS_PATH
else:
    raise FileNotFoundError(
        "Neither orders_with_escalation_scores.csv nor orders.csv was found."
    )

print("Orders file:", ORDERS_INPUT.resolve())


## 3. Connect to the Task 5 recommended RAG collection

Your Task 5 comparison showed:
- Fixed-size with overlap: Precision@3 = 0.444, Recall@3 = 0.917
- Sentence-based: Precision@3 = 0.500, Recall@3 = 1.000

Sentence-based chunking performed better on both reported averages, so this agent uses `nykaa_sentence_chunks`.

In [ ]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

try:
    policy_collection = chroma_client.get_collection(COLLECTION_NAME)
except Exception as exc:
    raise RuntimeError(
        f"Could not open ChromaDB collection '{COLLECTION_NAME}'. "
        "Run Task 3 first and check the collection name."
    ) from exc

embedding_model = SentenceTransformer(MODEL_NAME)

print("RAG collection loaded.")
print("Number of indexed chunks:", policy_collection.count())
print("Embedding model:", MODEL_NAME)

if policy_collection.count() == 0:
    raise ValueError("The selected RAG collection is empty.")


## 4. Load orders and recreate the Task 6 lookup tool

In [ ]:
orders_df = pd.read_csv(ORDERS_INPUT)

REQUIRED_COLUMNS = {
    "record_id", "status", "order_value_inr",
    "days_since_created", "delayed_shipment"
}
missing_columns = REQUIRED_COLUMNS - set(orders_df.columns)
if missing_columns:
    raise ValueError(f"Orders file is missing columns: {sorted(missing_columns)}")

orders_df["record_id"] = orders_df["record_id"].astype(str).str.strip()
orders_df["status"] = orders_df["status"].astype(str).str.strip()
orders_df["order_value_inr"] = pd.to_numeric(orders_df["order_value_inr"], errors="raise")
orders_df["days_since_created"] = pd.to_numeric(orders_df["days_since_created"], errors="raise")
orders_df["delayed_shipment"] = (
    orders_df["delayed_shipment"].astype(str).str.strip().str.lower()
    .map({"true": True, "false": False})
)

if orders_df["record_id"].duplicated().any():
    raise ValueError("record_id values must be unique.")
if orders_df["delayed_shipment"].isna().any():
    raise ValueError("delayed_shipment must contain only True/False.")

DELAY_WEIGHT = 0.70
RECENCY_WEIGHT = 0.30
MAX_ORDER_AGE_DAYS = 30

def calculate_escalation_score(order):
    delayed_signal = 1.0 if bool(order["delayed_shipment"]) else 0.0
    recency_normalized = float(order["days_since_created"]) / MAX_ORDER_AGE_DAYS
    score = DELAY_WEIGHT * delayed_signal + RECENCY_WEIGHT * recency_normalized
    return round(float(np.clip(score, 0.0, 1.0)), 4)

orders_df["escalation_score"] = orders_df.apply(calculate_escalation_score, axis=1)
ESCALATION_THRESHOLD = float(orders_df["escalation_score"].quantile(0.80))

def check_order_status(record_id: str) -> dict:
    """Look up an order and return status, value, and its 0–1 escalation score."""
    if record_id is None or not str(record_id).strip():
        return {"found": False, "record_id": record_id, "error": "A non-empty order ID is required."}

    requested_id = str(record_id).strip().upper()
    matches = orders_df[orders_df["record_id"].str.upper() == requested_id]

    if matches.empty:
        return {
            "found": False,
            "record_id": requested_id,
            "error": f"No order found for {requested_id}."
        }

    order = matches.iloc[0].to_dict()
    score = calculate_escalation_score(order)

    return {
        "found": True,
        "record_id": order["record_id"],
        "status": order["status"],
        "order_value_inr": float(order["order_value_inr"]),
        "escalation_score": score,
        "escalation_threshold": round(ESCALATION_THRESHOLD, 4),
        "requires_escalation": score >= ESCALATION_THRESHOLD
    }

print("Orders loaded:", len(orders_df))
print("Task 6 escalation threshold:", round(ESCALATION_THRESHOLD, 4))
print("Lookup function test:", check_order_status(orders_df.iloc[0]["record_id"]))


## 5. Define the LangGraph state

The state is the shared information passed between nodes. It stores the customer's question, detected intent, extracted order ID, retrieved policy context, order lookup result, and final answer.

In [ ]:
class SupportState(TypedDict, total=False):
    query: str
    intent: Literal["policy", "order"]
    record_id: Optional[str]
    retrieved_context: list
    order_result: dict
    answer: str
    sources: list
    route_taken: str

print("SupportState schema defined.")


## 6. Node 1 — Classify the customer's intent

For this first version, routing uses transparent rules:
- If an order ID such as `ORD1014` appears, classify it as an order lookup.
- If the query contains order-tracking/status language, classify it as an order lookup.
- Otherwise, route it to the policy knowledge base.

This is an intent-routing baseline. It can be replaced with a trained classifier or an LLM classifier in a later iteration.

In [ ]:
ORDER_ID_PATTERN = re.compile(r"\bORD\d{4,}\b", flags=re.IGNORECASE)

ORDER_INTENT_KEYWORDS = {
    "order status", "track my order", "track order",
    "where is my order", "check my order", "order update"
}

def classify_intent(state: SupportState) -> dict:
    query = state["query"].strip()
    query_lower = query.lower()

    id_match = ORDER_ID_PATTERN.search(query)
    has_order_language = any(term in query_lower for term in ORDER_INTENT_KEYWORDS)

    if id_match or has_order_language:
        intent = "order"
        record_id = id_match.group(0).upper() if id_match else None
    else:
        intent = "policy"
        record_id = None

    print(f"[classify_intent] intent={intent}, record_id={record_id}")
    return {
        "intent": intent,
        "record_id": record_id,
        "route_taken": intent
    }


## 7. Node 2 — Retrieve policy context using RAG

This node embeds the question, searches the sentence-based ChromaDB collection, and returns the top matching chunks with their source filenames.

The response composer will use only this retrieved text. It will not invent policy details.

In [ ]:
def retrieve_policy(state: SupportState) -> dict:
    query = state["query"]

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist()

    result = policy_collection.query(
        query_embeddings=[query_embedding],
        n_results=min(TOP_K, policy_collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    documents = result.get("documents", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]

    retrieved_context = []
    for document, metadata, distance in zip(documents, metadatas, distances):
        retrieved_context.append({
            "text": document,
            "source": metadata.get("source", "unknown"),
            "similarity": round(1 - float(distance), 4)
        })

    print(f"[retrieve_policy] retrieved {len(retrieved_context)} chunks")
    return {"retrieved_context": retrieved_context}


## 8. Node 3 — Look up an order

This node calls the Task 6 `check_order_status()` function. If the customer asks for an order status but does not provide an order ID, the node returns a request for the missing ID.

In [ ]:
def lookup_order(state: SupportState) -> dict:
    record_id = state.get("record_id")

    if not record_id:
        result = {
            "found": False,
            "error": "Please provide your order ID (for example, ORD1014) so I can check its status."
        }
    else:
        result = check_order_status(record_id)

    print("[lookup_order] lookup completed; found =", result.get("found"))
    return {"order_result": result}


## 9. Node 4 — Compose the final response

The node checks which branch ran:
- For order queries, it formats the returned order record.
- For policy queries, it presents the retrieved knowledge-base text and source.
- If no useful policy text was retrieved, it falls back rather than fabricating an answer.

This is deterministic response composition for the required `MOCK_LLM` environment.

In [ ]:
def compose_response(state: SupportState) -> dict:
    if state.get("intent") == "order":
        result = state.get("order_result", {})

        if not result.get("found"):
            answer = result.get("error", "I could not find that order.")
            sources = []
        else:
            escalation_text = (
                "This order is recommended for human support review."
                if result["requires_escalation"]
                else "This order does not currently meet the project's escalation threshold."
            )
            answer = (
                f"Order {result['record_id']} has status: {result['status']}. "
                f"Order value: ₹{result['order_value_inr']:,.2f}. "
                f"Escalation score: {result['escalation_score']:.2f}. "
                f"{escalation_text}"
            )
            sources = ["orders dataset (synthetic project data)"]

    else:
        contexts = state.get("retrieved_context", [])
        useful_contexts = [
            item for item in contexts
            if item.get("text") and item.get("source") != "unknown"
        ]

        if not useful_contexts:
            answer = "I don't know based on the available policy documents."
            sources = []
        else:
            # Show the retrieved policy passages; no unsupported claims are added.
            answer = (
                "According to the available project policy documents, the relevant "
                "information is:\n\n"
                + "\n\n".join(
                    f"[{item['source']}] {item['text']}"
                    for item in useful_contexts
                )
            )
            sources = list(dict.fromkeys(item["source"] for item in useful_contexts))

    print("[compose_response] final answer created")
    return {"answer": answer, "sources": sources}


## 10. Build the LangGraph workflow with a conditional edge

The graph is:

`START → classify_intent → (conditional) → retrieve_policy OR lookup_order → compose_response → END`

The conditional edge is genuine: it selects a different execution path based on the `intent` stored in the shared state.

In [ ]:
def route_after_classification(state: SupportState) -> Literal["policy", "order"]:
    if state.get("intent") == "order":
        return "order"
    return "policy"

builder = StateGraph(SupportState)

builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_policy", retrieve_policy)
builder.add_node("lookup_order", lookup_order)
builder.add_node("compose_response", compose_response)

builder.add_edge(START, "classify_intent")

builder.add_conditional_edges(
    "classify_intent",
    route_after_classification,
    {
        "policy": "retrieve_policy",
        "order": "lookup_order"
    }
)

builder.add_edge("retrieve_policy", "compose_response")
builder.add_edge("lookup_order", "compose_response")
builder.add_edge("compose_response", END)

support_agent = builder.compile()

print("PASS: LangGraph compiled.")
print("Nodes:", list(builder.nodes.keys()))
print("Conditional routing: policy → retrieve_policy; order → lookup_order")


## 11. Test Route A — Policy question goes to RAG

In [ ]:
policy_test = {
    "query": "How long does a cash-on-delivery refund take?"
}

policy_result = support_agent.invoke(policy_test)

print("\nROUTE:", policy_result.get("route_taken"))
print("ANSWER:")
print(policy_result.get("answer"))
print("SOURCES:", policy_result.get("sources"))

assert policy_result.get("intent") == "policy"
assert len(policy_result.get("retrieved_context", [])) > 0
assert policy_result.get("answer")
print("\nPASS: Policy question was routed to RAG.")


## 12. Test Route B — Order query goes to the order lookup tool

In [ ]:
# Choose an actual record ID from the loaded dataset.
sample_record_id = str(orders_df.iloc[0]["record_id"])

order_test = {
    "query": f"Can you check the status of order {sample_record_id}?"
}

order_result = support_agent.invoke(order_test)

print("\nROUTE:", order_result.get("route_taken"))
print("ORDER LOOKUP RESULT:", order_result.get("order_result"))
print("ANSWER:")
print(order_result.get("answer"))
print("SOURCES:", order_result.get("sources"))

assert order_result.get("intent") == "order"
assert order_result.get("order_result", {}).get("found") is True
assert order_result.get("answer")
print("\nPASS: Order query was routed to check_order_status.")


## 13. Test missing order ID and inspect graph structure

In [ ]:
missing_id_test = {
    "query": "Please check my order status."
}
missing_id_result = support_agent.invoke(missing_id_test)

print(missing_id_result["answer"])
assert missing_id_result["intent"] == "order"
assert missing_id_result["order_result"]["found"] is False
print("PASS: Agent asks for an order ID instead of inventing an order.")

# Display the compiled graph structure (Mermaid text).
print("\nLANGGRAPH STRUCTURE")
print(support_agent.get_graph().draw_mermaid())


## 14. Save Task 7 demonstration transcripts

The saved JSON file records the two required successful routes and the missing-order-ID test. These are useful evidence to retain in the final GitHub repository.

In [ ]:
import json

transcript = {
    "task": "Task 7 — LangGraph conditional routing",
    "policy_route": {
        "query": policy_test["query"],
        "intent": policy_result.get("intent"),
        "sources": policy_result.get("sources"),
        "answer": policy_result.get("answer")
    },
    "order_route": {
        "query": order_test["query"],
        "intent": order_result.get("intent"),
        "order_result": order_result.get("order_result"),
        "answer": order_result.get("answer")
    },
    "missing_order_id_route": {
        "query": missing_id_test["query"],
        "intent": missing_id_result.get("intent"),
        "answer": missing_id_result.get("answer")
    }
}

TRANSCRIPT_PATH = PROJECT_DIR / "task7_langgraph_transcripts.json"
TRANSCRIPT_PATH.write_text(
    json.dumps(transcript, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Transcript saved to:", TRANSCRIPT_PATH.resolve())
print("PASS: Both conditional routes were demonstrated.")
print("PASS: Four LangGraph nodes were created.")
print("PASS: Task 7 demonstration completed.")
